# 01 · Dataset inspection

Goal: understand the archive **before** training, and confirm it preserves
exercise, quality label, participant id and camera view.

Dataset: Mendeley *Multi-View Raw Video Dataset*, version 3
(`Dataset Exercise Quality-wise.zip`, CC BY 4.0).

Nothing is extracted permanently here - the zip is read as an archive so the
inspection is fast.

## Shared setup (run first)

This notebook uses the modules in `ml/`. Upload them once to the Colab runtime:

1. Click the folder icon in the left sidebar (Files).
2. Click **Upload to session storage** and select `extract_pose.py`,
   `fitness_features.py`, `build_dataset.py`, `train_baseline.py`,
   `train_temporal.py`, `export_onnx.py` from `F:\Fitness Coach\ml`.
3. Run the cell below.

The dataset zip is expected at
`/content/drive/MyDrive/Dataset Exercise Quality-wise.zip` (Google Drive).

In [ ]:
import os, sys, subprocess, json, glob
from pathlib import Path

ROOT = Path('/content/fitness')
ROOT.mkdir(parents=True, exist_ok=True)

# The uploaded modules land in /content; mirror them into the work root.
for f in ['extract_pose.py', 'fitness_features.py', 'build_dataset.py', 'train_baseline.py',
          'experiment_generalization.py', 'experiment_clip.py', 'experiment_repmetry.py',
          'train_final.py', 'train_temporal.py', 'export_onnx.py']:
    src = Path('/content') / f
    if src.exists():
        dst = ROOT / f
        if not dst.exists() or src.stat().st_mtime > dst.stat().st_mtime:
            dst.write_bytes(src.read_bytes())

sys.path.insert(0, str(ROOT))
print('work root :', ROOT)
print('modules   :', sorted(p.name for p in ROOT.glob('*.py')))
print('cuda      :', subprocess.run('nvidia-smi -L', shell=True, capture_output=True, text=True).stdout.strip() or 'no GPU')

## 1. Mount Drive and locate the archive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ZIP = '/content/drive/MyDrive/Dataset Exercise Quality-wise.zip'
import os
print('zip found:', os.path.exists(ZIP), round(os.path.getsize(ZIP)/1e9, 2), 'GB')

## 2. List the archive and infer the folder schema

In [ ]:
import zipfile, collections, os, re
z = zipfile.ZipFile(ZIP)
names = z.namelist()
files = [n for n in names if not n.endswith('/')]
print('entries:', len(names), '| files:', len(files))
print('extensions:', collections.Counter(os.path.splitext(n)[1].lower() for n in files).most_common())
print()
print('--- raw sample (first 12 entries) ---')
for n in names[:12]:
    print(' ', n)
print()
depths = collections.Counter(n.count('/') for n in names)
print('path depth distribution:', dict(sorted(depths.items())))

## 3. Parse the schema

Every path looks like
`Dataset Exercise Quality-wise/<exercise>/<quality>/<view>/subject_XXX_<exercise>_<quality>_<view>.mp4`.
The code below **derives** the fields and then verifies them against the
filename instead of assuming they agree.

In [ ]:
import pandas as pd, re
rows = []
for n in files:
    if not n.lower().endswith('.mp4'):
        continue
    parts = n.split('/')
    rows.append({'path': n, 'exercise': parts[1], 'quality': parts[2], 'view': parts[3],
                 'filename': parts[-1]})
df = pd.DataFrame(rows)
df['subject'] = df['filename'].str.extract(r'subject_(\d+)')
pattern = re.compile(r'subject_(\d+)_(.+)_(good|bad)_(front|side|diagonal)\.mp4$')
bad = [f for f in df['filename'] if not pattern.match(f)]
print('videos:', len(df))
print('exercises:', sorted(df.exercise.unique()))
print('qualities:', sorted(df.quality.unique()))
print('views    :', sorted(df.view.unique()))
print('subjects :', df.subject.nunique(), sorted(df.subject.unique()))
print('filenames that do not match the expected pattern:', len(bad), bad[:5])

# folder label == filename label? folder view == filename view?
mm = df['filename'].str.extract(pattern)
mismatch = ((mm[2] != df.quality) | (mm[3] != df.view) | (mm[1] != df.exercise)).sum()
print('folder/filename label mismatches:', int(mismatch))

## 4. Class and view distribution per exercise

In [ ]:
pivot = df.pivot_table(index='exercise', columns=['quality'], values='filename', aggfunc='count', fill_value=0)
print(pivot)
print()
print(df.pivot_table(index=['exercise','view'], columns='quality', values='filename', aggfunc='count', fill_value=0))
print()
print('videos per exercise x view x quality:')
print(df.groupby(['exercise','view','quality']).size().unstack().describe().round(1))

## 5. Participant distribution (this drives the split later)

In [ ]:
sub = df.groupby(['exercise','quality']).subject.nunique().unstack()
print(sub)
print()
# how many clips does each participant contribute per exercise?
per_subject = df.groupby(['subject','exercise']).size().unstack(fill_value=0)
print(per_subject.describe().round(2))
print()
print('participants missing for some exercise:', per_subject[(per_subject == 0).any(axis=1)].index.tolist())

## 6. Integrity checks

Read the video headers (no full decode) to catch corrupted or zero-length
files, and confirm resolution/fps/duration before committing to a sampling
rate.

In [ ]:
import cv2, tempfile, numpy as np
probe = []
tmp = tempfile.mkdtemp()
for n in df['path'].sample(min(60, len(df)), random_state=0):
    with z.open(n) as src:
        data = src.read()
    p = os.path.join(tmp, 'v.mp4')
    open(p, 'wb').write(data)
    cap = cv2.VideoCapture(p)
    fps = cap.get(cv2.CAP_PROP_FPS)
    nf = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    w = cap.get(cv2.CAP_PROP_FRAME_WIDTH); h = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    ok, _ = cap.read()
    cap.release()
    probe.append({'fps': round(fps, 2), 'frames': int(nf), 'w': int(w), 'h': int(h),
                  'sec': round(nf / fps, 1) if fps else None, 'readable': bool(ok)})
pr = pd.DataFrame(probe)
print(pr.describe().round(2))
print()
print('fps values:', pr.fps.value_counts().to_dict())
print('unreadable:', int((~pr.readable).sum()), '| zero-frame:', int((pr.frames <= 0).sum()))

## 7. Archive selection decision

Conclusion from this inspection:

* The **Quality-wise** archive is fully labelled: `exercise / quality / view`
  folders *and* redundant labels in the filename, cross-checked above.
* Participant ids are present for all 26 participants, so a subject-level
  split is possible.
* The three target exercises (`squat`, `bicep_curl`, `push_up`) each have
  ~25 participants x 3 views x 2 qualities.

**Selected archive: `Dataset Exercise Quality-wise.zip`.** The other two
archives are the same videos re-foldered and are not needed.